# MultiDay_Reference — per-subject reference cells
Cells are segmented **once per subject** on a reference volume and keep a global `cell_id`. Each FLIM session is
registered to this reference in `AnalyzeTCSPC_XYTZTile.ipynb` (Step 2: `use_subject_reference = True` → Step 2b), and
the session's own TCSPC data give every cell's lifetime.

1. **Reference volume** — a dedicated intensity stack (finer z, larger area than the daily FLIM FOV, more averaging;
   Fluoview-stitched ZYX TIFF), or, until it exists, a session's stitched volume: preferably the Olympus red channel
   (`<base>_stitched_CH1.tif`, complete and without the diffuse green background), else `<base>_stitched_intensity.tif`.
   Do not use a session where Vista dropped planes without Olympus tiles to repair it (e.g. 11-02).
2. **Cellpose tuning** on a crop (settings grid).
3. **Full segmentation** → `reference/reference_labels_auto.tif`.
4. **Curation in napari** → `reference/reference_labels.tif` (IDs frozen from here on; new cells get new IDs).
5. **Across sessions** — after Steps 2b–16 ran for the sessions: `cells_long.csv` + plots.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'   # prevent OpenMP conflict between MKL and torch on Windows
import torch   # must be imported before numpy/scipy to avoid OpenMP runtime clash

import sys
import json
import importlib
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

sys.path.insert(0, 'X:/FLIM_2026')
import multiday as md
importlib.reload(md)
print(f'torch {torch.__version__}  GPU available: {torch.cuda.is_available()}')

## 1. Reference volume
The reference is recorded as a path + voxel size in `reference/reference_meta.json` (the TIFF is not copied).
Changing it after curation would invalidate `reference_labels.tif`, so that needs `replace_reference = True`.

In [ ]:
subject_id   = '336'
subject_root = r'D:\Peredox_FLIM\subjects'
# dedicated reference stack (ZYX TIFF) + its voxel size, or a session's stitched volume (interim reference).
# Interim: the 2026-11-06 session's Olympus red channel. Not 11-02 (..._X+0.50_Y+0.80_..._Z186): Vista dropped
# 8 planes there and, without Olympus tiles, they cannot be put back, so its stitch is unreliable.
ref_path     = r'D:\Peredox_FLIM\XYZTTile1_336_X0.4Y-2.8_Decay128_512x512_Z210_Ch1_T1_P1_S1\XYZTTile1_336_X0.4Y-2.8_Decay128_512x512_Z210_Ch1_T1_P1_S1_stitched_CH1.tif'
ref_spacing  = (4.0, 0.9944, 0.9944)   # (dz, dy, dx) µm
replace_reference = False

subject_dir = os.path.join(subject_root, subject_id)
P = md.subject_paths(subject_dir)
if os.path.exists(P['ref_meta']):
    _old = json.load(open(P['ref_meta']))
    if (_old['source'] != ref_path or list(_old['spacing_zyx_um']) != list(ref_spacing)) \
            and os.path.exists(P['ref_labels']) and not replace_reference:
        raise RuntimeError(f'Subject {subject_id} already has curated labels for {_old["source"]}; '
                           'set replace_reference = True to switch reference (labels must be redone).')
ref_meta = md.save_reference_meta(subject_dir, ref_path, ref_spacing, subject=subject_id)
ref_vol, _ = md.load_volume(ref_path, ref_spacing)
print(f'Reference {ref_vol.shape} (z, y, x) at {ref_spacing} µm  ->  {subject_dir}')

_mip = ref_vol.max(axis=0)
_xz  = ref_vol[:, ref_vol.shape[1] // 2]
fig, axs = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw=dict(width_ratios=[1, 1.4, 0.5]))
axs[0].imshow(_mip, cmap='gray', vmax=np.percentile(_mip, 99.5)); axs[0].set_title('max over z')
axs[1].imshow(_xz, cmap='gray', aspect=ref_spacing[0] / ref_spacing[2], vmax=np.percentile(_xz, 99.5))
axs[1].set_title('xz section (middle y)')
axs[2].plot(ref_vol.mean(axis=(1, 2)), np.arange(ref_vol.shape[0]) * ref_spacing[0])
axs[2].invert_yaxis(); axs[2].set_xlabel('mean intensity'); axs[2].set_ylabel('depth (µm)')
plt.tight_layout(); plt.show()

## 2. Cellpose tuning on a crop
Each row of the montage is one setting (outlines on the smoothed input), columns are z planes. Pick the setting whose
outlines follow the objects you want to track. Notes:
- `2d_stitch` segments each plane and links masks across z; `3d` is far slower at small diameters (minutes per crop).
- Lower `cellprob_threshold` grows/merges objects rather than adding many new ones on this data.
- On the 2026-11-02 session Cellpose found few objects at any setting: the bright objects are small (5–8 µm) and
  often split into horizontal bars (brain motion during line scanning). Expect to add cells by hand in step 4, or
  fine-tune Cellpose on curated planes later.

In [ ]:
crop_yx      = (600, 300)            # top-left (y, x) of the crop, px
crop_size    = 256
z_show       = None                  # planes in the montage; None = 4 evenly spaced
seg_sigma_um = (1.0, 0.7, 0.7)       # light smoothing of the Cellpose input (z, y, x) µm
grid = [
    dict(mode='2d_stitch', diameter_um=8,  cellprob_threshold=-1.5),
    dict(mode='2d_stitch', diameter_um=8,  cellprob_threshold=0.0),
    dict(mode='2d_stitch', diameter_um=12, cellprob_threshold=-1.5),
    dict(mode='2d_stitch', diameter_um=16, cellprob_threshold=-1.5),
]

_y0, _x0 = crop_yx
_crop = md.seg_input(ref_vol[:, _y0:_y0 + crop_size, _x0:_x0 + crop_size], ref_spacing, sigma_um=seg_sigma_um)
_zs   = z_show or [int(z) for z in np.linspace(0, _crop.shape[0] - 1, 6)[1:-1].round()]
grid_res = md.cellpose_grid(_crop, ref_spacing, grid)
md.plot_cellpose_grid(_crop, grid_res, _zs, title=f'crop y0={_y0} x0={_x0}  ({crop_size} px)')
plt.show()
pd.DataFrame([dict(**r['settings'], **{k: round(v, 1) for k, v in r['stats'].items()}, seconds=round(r['seconds']))
              for r in grid_res])

## 3. Full segmentation
Runs the chosen setting on the whole reference → `reference_labels_auto.tif` (+ `.json` with the settings).
This is the starting point of the curation; it is never used directly by the sessions.

In [ ]:
seg_settings = dict(mode='2d_stitch', diameter_um=8, cellprob_threshold=-1.5)   # chosen from the grid
rerun_seg    = False

_json = P['ref_labels_auto'].replace('.tif', '.json')
_want = dict(seg_settings, seg_sigma_um=list(seg_sigma_um), source=ref_path)
if os.path.exists(P['ref_labels_auto']) and os.path.exists(_json) and json.load(open(_json)) == _want and not rerun_seg:
    labels_auto = tifffile.imread(P['ref_labels_auto'])
    print(f'Reloaded {os.path.basename(P["ref_labels_auto"])}')
else:
    _inp = md.seg_input(ref_vol, ref_spacing, sigma_um=seg_sigma_um)
    labels_auto = md.segment_cellpose(_inp, ref_spacing, **seg_settings)
    tifffile.imwrite(P['ref_labels_auto'], labels_auto, compression='zlib')
    with open(_json, 'w') as f:
        json.dump(_want, f, indent=1)
    print(f'Saved -> {P["ref_labels_auto"]}')
print(md.mask_stats(labels_auto, ref_spacing))

## 4. Curation in napari
Opens the reference and the labels in napari (`napari-env`); the cell waits until the window is closed.
Start: `reference_labels.tif` if it exists (continue curating), else `reference_labels_auto.tif`.
- select the **cells** layer: paint `P`, erase `E`, fill `F`, pick a label `L`; the brush works in the current plane
  (use the z slider), or in 3D with *n edit dim* = 3
- new cell: set the label number to an unused ID (the layer starts at max ID + 1); delete a cell: fill it with 0;
  merge: fill one with the other's ID
- **Ctrl+Shift+S** saves to `reference_labels.tif`; closing the window saves too
- IDs are global and kept from here on: do not renumber; removed IDs are simply absent

In [ ]:
napari_env = r'X:\conda\envs\napari-env'
_start = P['ref_labels'] if os.path.exists(P['ref_labels']) else P['ref_labels_auto']
_env = dict(os.environ)
_env['PATH'] = ';'.join([napari_env, rf'{napari_env}\Library\mingw-w64\bin', rf'{napari_env}\Library\usr\bin',
                         rf'{napari_env}\Library\bin', rf'{napari_env}\Scripts', rf'{napari_env}\bin', _env['PATH']])
_env.pop('KMP_DUPLICATE_LIB_OK', None)
print(f'napari: {os.path.basename(_start)} -> {os.path.basename(P["ref_labels"])}   (close the window when done)')
_r = subprocess.run([rf'{napari_env}\python.exe', r'X:\FLIM_2026\napari_curate.py', ref_meta['source'], _start,
                     P['ref_labels'], ','.join(str(s) for s in ref_spacing)],
                    env=_env, capture_output=True, text=True)
print(_r.stdout[-2000:] or '(no output)')
if _r.returncode:
    print(_r.stderr[-3000:])

In [ ]:
# ── freeze the curated cells: table of IDs, volumes and centroids ─────────────
labels_ref = tifffile.imread(P['ref_labels']).astype(np.uint16)
cells_ref  = md.reference_cells_table(labels_ref, ref_spacing)
if os.path.exists(P['ref_cells']):
    _prev = set(pd.read_csv(P['ref_cells']).cell_id)
    _gone, _new = sorted(_prev - set(cells_ref.cell_id)), sorted(set(cells_ref.cell_id) - _prev)
    print(f'vs previous table: {len(_new)} new IDs, {len(_gone)} removed IDs' + (f'  removed: {_gone[:20]}' if _gone else ''))
cells_ref.to_csv(P['ref_cells'], index=False)
print(f'{len(cells_ref)} reference cells (IDs {cells_ref.cell_id.min()}–{cells_ref.cell_id.max()})  ->  {P["ref_cells"]}')
print(md.mask_stats(labels_ref, ref_spacing))

## 5. Across sessions
Collects every session registered in `sessions.json` (Step 2b adds them) whose Steps 14–16 have run: per-cell
lifetimes, phasor, tav statistics and the registration QC → `cells_long.csv` (one row per session × cell).
Only cells flagged `ok` are plotted by default.

In [ ]:
long = md.collect_subject(subject_dir)
if len(long):
    md.plot_subject(long, value='aw_LT', flags=('ok',), min_sessions=2, title=f'subject {subject_id}: aw_LT')
    plt.show()
    if 'median_tav' in long:
        md.plot_subject(long, value='median_tav', flags=('ok',), min_sessions=2,
                        title=f'subject {subject_id}: median tav')
        plt.show()
    display(long.groupby(['date', 'flag']).size().unstack(fill_value=0))